<a href="https://colab.research.google.com/github/EthanIsaac/valorant-voiceline-qa/blob/main/model_development/notebooks/03_speech_to_text_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Speech-to-Text: fine-tuning Whisper on VALORANT voice lines

**Project:** VALORANT Voice-Line QA & Localization Assistant (Riot Games)
**Pipeline being improved:** `automatic-speech-recognition` (notebook 02 compared three pre-trained models and selected Whisper).

**Why fine-tune:** in notebook 02 most of Whisper's errors were **game vocabulary**: site callouts ("C" heard as *see*), game terms (*spike*) and agent names (*Reyna* → *Rayna*).
We have 8.6k English training lines with transcripts, the classic case for adapting a pre-trained model to its domain. It is also realistic for Riot: new voice lines reuse the same game vocabulary.

**Two sizes are fine-tuned:** `whisper-base.en` (the most accurate in notebook 02) and `whisper-tiny.en` (about 2× faster on CPU).
If fine-tuning brings tiny close to base, the app can be faster without losing accuracy.

**Notebook outline**
- Step 0: Setup (packages, seed, output folder, Hugging Face token)
- Step 1: Load the English clips of all three splits (same split as notebooks 01 and 02)
- Step 2: Flag test lines that also appear in training (to measure memorisation vs. generalisation)
- Step 3: Score the pre-trained models (the baseline)
- Step 4: Fine-tune both models
- Step 5: Score the fine-tuned models and measure CPU runtime
- Step 6: Compare and pick the speech-to-text model for the app
- Step 7: Upload the selected model to Hugging Face (if it is a fine-tuned one)
- Save and download the results (with SHA-256 fingerprints)

All results are written to `results/speech_to_text_finetuning/`.

> Runtime: **GPU** (Runtime → Change runtime type → T4 GPU).

# Step 0: Setup

In [1]:
# Same transformers constraint as the course notebooks; jiwer computes the word error rate.
# Expected: a pip "dependency conflicts" message about diffusers/gradio needing a newer
# huggingface-hub. Those packages are not used here, so it can be ignored.
%pip install -q "transformers==4.57.6" accelerate "jiwer==4.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import hashlib
import importlib.metadata
import io
import json
import os
import shutil
import time
import urllib.request

import jiwer
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import transformers
from IPython.display import display
from transformers import set_seed

# One seed for everything (Python, NumPy, PyTorch), so reruns give the same results
SEED = 42
set_seed(SEED)

# Every result file produced by this notebook goes in this folder
RESULTS_DIR = "results/speech_to_text_finetuning"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Publishing the model to Hugging Face (Step 7) is off unless the environment variable UPLOAD_HF_MODELS is "true"
# (for the author: run `%env UPLOAD_HF_MODELS=true` before Run all). Anyone else runs the whole notebook without
# a token: the upload is skipped and the Colab secret is never requested.
UPLOAD_HF_MODELS = os.environ.get("UPLOAD_HF_MODELS", "").strip().lower() == "true"
if UPLOAD_HF_MODELS:
    # Read explicitly here: if the secret is granted after the first Hub download, the library would not see it
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Upload to Hugging Face: on (token read from the Colab secret HF_TOKEN)")
else:
    print("Upload to Hugging Face: off (set UPLOAD_HF_MODELS=true to publish the model in Step 7)")

JIWER_VERSION = importlib.metadata.version("jiwer")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers {transformers.__version__} | torch {torch.__version__} | jiwer {JIWER_VERSION} | device: {device}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Upload to Hugging Face: off (set UPLOAD_HF_MODELS=true to publish the model in Step 7)
transformers 4.57.6 | torch 2.11.0+cu130 | jiwer 4.0.0 | device: cuda
GPU: Tesla T4


# Step 1: Load the English clips of all three splits

The **same split as notebooks 01 and 02** (`results/agent_classifier/splits.csv`), downloaded from the project's GitHub repository at a fixed commit and verified with its SHA-256 fingerprint:
- **train**: used for fine-tuning
- **validation**: used to pick the best epoch and the final model
- **test**: used once, for the final scores

Only clips flagged **`english_only`** are used (speech, no foreign words or translator notes), with the cleaned `spoken_text` as the target transcript.
The audio comes from the same pinned dataset revision (about 2 GB download).

In [3]:
REPO_COMMIT = "2837dcb342749a346d4370e63545c575ba5c2a07"   # commit with the notebook 01 results
SPLITS_URL = f"https://raw.githubusercontent.com/EthanIsaac/valorant-voiceline-qa/{REPO_COMMIT}/results/agent_classifier/splits.csv"
SPLITS_SHA256 = "72b658b97258de7bc3e77a2b88908080dd46be76d0162c8905cceff2c903dd7d"   # from results/agent_classifier/SHA256SUMS

splits_bytes = urllib.request.urlopen(SPLITS_URL).read()
assert hashlib.sha256(splits_bytes).hexdigest() == SPLITS_SHA256, "splits.csv does not match notebook 01"
splits = pd.read_csv(io.BytesIO(splits_bytes))

clips = splits[splits["english_only"]].reset_index(drop=True)
print(clips["split"].value_counts().loc[["train", "validation", "test"]].to_string())

split
train         8611
validation    1059
test          1070


In [4]:
from huggingface_hub import hf_hub_download, list_repo_files

DATASET = "NgThVinh/ValorantAgentVoiceLines"
REVISION = "c2e0d3493f44da5be07090d6ebb9b69ebbcb2060"   # same pinned commit as notebooks 01 and 02
TARGET_SR = 16_000   # Whisper expects 16 kHz audio

parquet_files = sorted(
    f for f in list_repo_files(DATASET, repo_type="dataset", revision=REVISION) if f.endswith(".parquet")
)
audio_bytes = {}
for file in parquet_files:
    agent = file.split("/")[0]
    agent_df = pd.read_parquet(hf_hub_download(DATASET, file, repo_type="dataset", revision=REVISION))
    wanted = set(clips.loc[clips["agent"] == agent, "audio_name"])
    for name, audio in zip(agent_df["audio_name"], agent_df["audio_file"]):
        if name in wanted:
            audio_bytes[(agent, name)] = audio["bytes"]


def load_waveform(wav_bytes):
    """WAV bytes -> mono float32 waveform at 16 kHz. Not trimmed: every word must be heard."""
    waveform, sr = sf.read(io.BytesIO(wav_bytes), dtype="float32")
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)
    return librosa.resample(waveform, orig_sr=sr, target_sr=TARGET_SR)


start = time.perf_counter()
clips["waveform"] = [load_waveform(audio_bytes[(a, n)]) for a, n in zip(clips["agent"], clips["audio_name"])]
del audio_bytes
data = {split: clips[clips["split"] == split].reset_index(drop=True) for split in ["train", "validation", "test"]}
print(f"Decoded {len(clips)} clips in {time.perf_counter() - start:.0f}s")

astra/train-00000-of-00001.parquet:   0%|          | 0.00/83.9M [00:00<?, ?B/s]

breach/train-00000-of-00001.parquet:   0%|          | 0.00/65.3M [00:00<?, ?B/s]

brimstone/train-00000-of-00001.parquet:   0%|          | 0.00/58.7M [00:00<?, ?B/s]

chamber/train-00000-of-00001.parquet:   0%|          | 0.00/69.8M [00:00<?, ?B/s]

clove/train-00000-of-00001.parquet:   0%|          | 0.00/89.3M [00:00<?, ?B/s]

...

tejo/train-00000-of-00001.parquet:   0%|          | 0.00/51.4M [00:00<?, ?B/s]

viper/train-00000-of-00001.parquet:   0%|          | 0.00/73.9M [00:00<?, ?B/s]

vyse/train-00000-of-00001.parquet:   0%|          | 0.00/69.7M [00:00<?, ?B/s]

yoru/train-00000-of-00001.parquet:   0%|          | 0.00/62.2M [00:00<?, ?B/s]

Decoded 10740 clips in 73s


# Step 2: Flag lines that also appear in training

The split keeps all takes of a line **by the same agent** together, but many lines are shared **between agents** (callouts such as *"Let's rush A."*).
For the agent classifier that made a fair test (same words, different voice). For speech-to-text it is a trap: a fine-tuned model could score well on those lines simply by **memorising the script**.

So each validation and test line is flagged `seen_in_train` if the same text (after normalisation) appears anywhere in the training set.
Scores on the **unseen** lines measure real generalisation; those are the scores used to pick the final model.

Same text normalisation as notebook 02 (Whisper's English normaliser) for all scoring.

In [5]:
from transformers.models.whisper.english_normalizer import EnglishTextNormalizer
from transformers import AutoTokenizer

normalizer = EnglishTextNormalizer(AutoTokenizer.from_pretrained("openai/whisper-base.en").english_spelling_normalizer)
for split in data:
    data[split]["reference"] = data[split]["spoken_text"].map(normalizer)

train_lines = set(data["train"]["reference"])
for split in ["validation", "test"]:
    data[split]["seen_in_train"] = data[split]["reference"].isin(train_lines)
    print(f"{split:<11} {len(data[split])} lines, {data[split]['seen_in_train'].mean():.1%} also appear in training")

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

validation  1059 lines, 50.7% also appear in training
test        1070 lines, 53.6% also appear in training


# Step 3: Score the pre-trained models (the baseline)

Both models transcribe validation and test through the same `pipeline("automatic-speech-recognition")` call as notebook 02 (one clip at a time, greedy decoding).
The pre-trained test scores should therefore match notebook 02 exactly, which is a consistency check between the notebooks.

Metrics as in notebook 02: **WER** (word error rate, lower is better) and **exact match** (share of lines transcribed perfectly, i.e. lines that would pass QA automatically), overall and on lines unseen / seen in training.

In [6]:
from transformers import pipeline

CANDIDATES = {
    "whisper_tiny": "openai/whisper-tiny.en",
    "whisper_base": "openai/whisper-base.en",
}


def transcribe(asr, waveforms):
    """One clip at a time, like the app."""
    return [asr({"raw": w, "sampling_rate": TARGET_SR})["text"].strip() for w in waveforms]


def score(df, raw_transcripts):
    """WER and exact match overall and on lines unseen / seen in training."""
    hypotheses = np.array([normalizer(t) for t in raw_transcripts], dtype=object)
    references = df["reference"].to_numpy(dtype=object)
    exact = hypotheses == references
    seen = df["seen_in_train"].to_numpy()
    scores = {}
    for subset, mask in [("all", np.ones(len(df), bool)), ("unseen", ~seen), ("seen", seen)]:
        if not mask.any():
            scores[f"wer_{subset}"], scores[f"exact_match_{subset}"] = np.nan, np.nan
            continue
        scores[f"wer_{subset}"] = jiwer.wer(list(references[mask]), list(hypotheses[mask]))
        scores[f"exact_match_{subset}"] = float(exact[mask].mean())
    return scores, list(hypotheses)


results = {}        # one row per (model, pre-trained or fine-tuned)
transcripts = data["test"][["agent", "audio_name", "spoken_text", "reference", "seen_in_train"]].copy()

for key, model_id in CANDIDATES.items():
    asr = pipeline("automatic-speech-recognition", model=model_id, device=0 if device == "cuda" else -1)
    for split in ["validation", "test"]:
        raw = transcribe(asr, data[split]["waveform"])
        scores, _ = score(data[split], raw)
        for name, value in scores.items():
            results.setdefault(f"{key}_pretrained", {"model_id": model_id, "fine_tuned": False})[f"{split}_{name}"] = value
        if split == "test":
            transcripts[f"{key}_pretrained"] = raw
    print(f"{key}_pretrained  validation WER {results[f'{key}_pretrained']['validation_wer_all']:.3f} | "
          f"test WER {results[f'{key}_pretrained']['test_wer_all']:.3f}")
    del asr
    torch.cuda.empty_cache()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


whisper_base_pretrained  validation WER 0.162 | test WER 0.118


# Step 4: Fine-tune both models

| Setting | Value | Why |
|---|---|---|
| Training data | 8.6k English training lines (audio + `spoken_text`) | The target keeps casing and punctuation, so the model still writes natural subtitles |
| What is trained | The whole model (encoder and decoder) | The standard Whisper fine-tuning recipe |
| Epochs | 3 | Whisper is already a strong recogniser; a few passes adapt it to the vocabulary, more would mainly memorise the script |
| Learning rate | 1e-5 (10% warm-up) | Low, standard value for Whisper: adapts without erasing what it learned from 680k hours of audio |
| Batch size | 16 | Fits the T4 GPU for both sizes |
| Mixed precision (fp16) | On | About 2× faster on the T4 |
| Best epoch | Lowest **validation WER** | The epoch is kept only if it helps on held-out lines |

Whisper always processes a 30-second window, so a batch can be padded without changing the result. Validation during training is therefore batched (faster); the final scores in Step 5 use the app's one-clip-at-a-time pipeline.

In [7]:
from dataclasses import dataclass

from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments, WhisperForConditionalGeneration,
                          WhisperProcessor)

NUM_EPOCHS = 3
LEARNING_RATE = 1e-5
BATCH_SIZE = 16


class SpeechLines(torch.utils.data.Dataset):
    """Waveform + target text; the log-Mel features are computed per batch to save memory."""
    def __init__(self, df):
        self.waveforms, self.texts = list(df["waveform"]), list(df["spoken_text"])

    def __len__(self):
        return len(self.waveforms)

    def __getitem__(self, i):
        return {"audio": self.waveforms[i], "text": self.texts[i]}


@dataclass
class WhisperCollator:
    processor: WhisperProcessor
    decoder_start_token_id: int

    def __call__(self, batch):
        features = self.processor.feature_extractor(
            [b["audio"] for b in batch], sampling_rate=TARGET_SR, return_tensors="pt")
        labels = self.processor.tokenizer([b["text"] for b in batch], padding=True, return_tensors="pt")
        # Padding positions are ignored by the loss (-100)
        label_ids = labels["input_ids"].masked_fill(labels["attention_mask"].ne(1), -100)
        # The model adds the start token itself; drop it from the labels
        if (label_ids[:, 0] == self.decoder_start_token_id).all():
            label_ids = label_ids[:, 1:]
        return {"input_features": features["input_features"], "labels": label_ids}


def make_compute_metrics(processor):
    def compute_metrics(eval_pred):
        pred_ids, label_ids = eval_pred.predictions, eval_pred.label_ids
        label_ids = np.where(label_ids == -100, processor.tokenizer.pad_token_id, label_ids)
        preds = [normalizer(t) for t in processor.batch_decode(pred_ids, skip_special_tokens=True)]
        refs = [normalizer(t) for t in processor.batch_decode(label_ids, skip_special_tokens=True)]
        return {"wer": jiwer.wer(refs, preds)}
    return compute_metrics


train_logs = {}
for key, model_id in CANDIDATES.items():
    print(f"\n=== Fine-tuning {key} ({model_id}) ===")
    set_seed(SEED)
    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperForConditionalGeneration.from_pretrained(model_id)

    args = Seq2SeqTrainingArguments(
        output_dir=f"checkpoints/{key}",
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.1,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        fp16=device == "cuda",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        save_total_limit=1,
        predict_with_generate=True,
        generation_max_length=128,
        logging_steps=50,
        dataloader_num_workers=2,   # compute audio features on the CPU while the GPU trains
        remove_unused_columns=False,
        seed=SEED,
        data_seed=SEED,
        report_to="none",
    )
    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=SpeechLines(data["train"]),
        eval_dataset=SpeechLines(data["validation"]),
        data_collator=WhisperCollator(processor, model.config.decoder_start_token_id),
        compute_metrics=make_compute_metrics(processor),
        processing_class=processor.feature_extractor,
    )
    train_output = trainer.train()
    train_logs[key] = trainer.state.log_history

    epochs = [(log["epoch"], log["eval_wer"]) for log in trainer.state.log_history if "eval_wer" in log]
    print("validation WER per epoch:", [(int(e), round(w, 4)) for e, w in epochs])
    results[f"{key}_finetuned"] = {
        "model_id": f"{model_id} (fine-tuned)",
        "fine_tuned": True,
        "best_epoch": int(min(epochs, key=lambda e: e[1])[0]),
        "train_runtime_s": train_output.metrics["train_runtime"],
    }

    # Keep the best model (with its processor, so a pipeline can load it); drop checkpoints, free GPU memory
    trainer.save_model(f"models/{key}")
    processor.save_pretrained(f"models/{key}")
    shutil.rmtree(f"checkpoints/{key}", ignore_errors=True)
    del trainer, model
    torch.cuda.empty_cache()

with open(f"{RESULTS_DIR}/training_logs.json", "w") as f:
    json.dump(train_logs, f, indent=2)


=== Fine-tuning whisper_tiny (openai/whisper-tiny.en) ===


Epoch,Training Loss,Validation Loss,Wer
1,0.201100,0.223279,0.088005
2,0.114400,0.208373,0.082648
3,0.076500,0.208431,0.084752


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 357, 366, 438, 532, 685, 705, 796, 930, 1058, 1220, 1267, 1279, 1303, 1343, 1377, 1391, 1635, 1782, 1875, 2162, 2361, 2488, 3467, 4008, 4211, 4600, 4808, 5299, 5855, 6329, 7203, 9609, 9959, 10563, 10786, 11420, 11709, 11907, 13163, 13697, 13700, 14808, 15306, 16410, 16791, 17992, 19203, 19510, 20724, 22305, 22935, 27007, 30109, 30420, 33409, 34949, 40283, 40493, 40549, 47282, 49146, 50257, 50357, 50358, 50359, 50360, 50361]}. You are seeing this warning because you've set gener

validation WER per epoch: [(1, 0.088), (2, 0.0826), (3, 0.0848)]

=== Fine-tuning whisper_base (openai/whisper-base.en) ===


Epoch,Training Loss,Validation Loss,Wer
1,0.148800,0.174421,0.062751
2,0.081900,0.166151,0.057968
3,0.048900,0.177395,0.056820


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 357, 366, 438, 532, 685, 705, 796, 930, 1058, 1220, 1267, 1279, 1303, 1343, 1377, 1391, 1635, 1782, 1875, 2162, 2361, 2488, 3467, 4008, 4211, 4600, 4808, 5299, 5855, 6329, 7203, 9609, 9959, 10563, 10786, 11420, 11709, 11907, 13163, 13697, 13700, 14808, 15306, 16410, 16791, 17992, 19203, 19510, 20724, 22305, 22935, 27007, 30109, 30420, 33409, 34949, 40283, 40493, 40549, 47282, 49146, 50257, 50357, 50358, 50359, 50360, 50361]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


validation WER per epoch: [(1, 0.0628), (2, 0.058), (3, 0.0568)]


# Step 5: Score the fine-tuned models and measure CPU runtime

The fine-tuned models are scored exactly like the baseline in Step 3 (same pipeline, one clip at a time).

Runtime is measured on **CPU** with the same `pipeline` call the app will use, for all four models: model loading time from a local folder (`cpu_load_s`) and time per clip over the first 100 test clips (`cpu_ms_per_clip`).
Fine-tuning does not change a model's size, so a fine-tuned model should run as fast as its pre-trained version.

In [8]:
for key in CANDIDATES:
    asr = pipeline("automatic-speech-recognition", model=f"models/{key}", device=0 if device == "cuda" else -1)
    for split in ["validation", "test"]:
        raw = transcribe(asr, data[split]["waveform"])
        scores, _ = score(data[split], raw)
        for name, value in scores.items():
            results[f"{key}_finetuned"][f"{split}_{name}"] = value
        if split == "test":
            transcripts[f"{key}_finetuned"] = raw
    print(f"{key}_finetuned  validation WER {results[f'{key}_finetuned']['validation_wer_all']:.3f} | "
          f"test WER {results[f'{key}_finetuned']['test_wer_all']:.3f}")
    del asr
    torch.cuda.empty_cache()

transcripts.to_csv(f"{RESULTS_DIR}/test_transcripts.csv", index=False)

Device set to use cuda:0
`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 357, 366, 438, 532, 685, 705, 796, 930, 1058, 1220, 1267, 1279, 1303, 1343, 1377, 1391, 1635, 1782, 1875, 2162, 2361, 2488, 3467, 4008, 4211, 4600, 4808, 5299, 5855, 6329, 7203, 9609, 9959, 10563, 10786, 11420, 11709, 11907, 13163, 13697, 13700, 14808, 15306, 16410, 16791, 17992, 19203, 19510, 20724, 22305, 22935, 27007, 30109, 30420, 33409, 34949, 40283, 40493, 40549, 47282, 49146, 50257, 50357, 50358, 50359, 50360, 50361], 'begin_suppress_tokens': [220, 50256]}. If this is not desired, please set these values explicitly.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.g

whisper_tiny_finetuned  validation WER 0.077 | test WER 0.060


Device set to use cuda:0


whisper_base_finetuned  validation WER 0.055 | test WER 0.043


In [9]:
from huggingface_hub import snapshot_download

speed_clips = data["test"].head(100)

for name in results:
    key, version = name.rsplit("_", 1)
    # Both versions load from a local folder, so only loading is timed (no network)
    source = snapshot_download(CANDIDATES[key]) if version == "pretrained" else f"models/{key}"

    start = time.perf_counter()
    asr = pipeline("automatic-speech-recognition", model=source, device=-1)   # device=-1 means CPU
    load_s = time.perf_counter() - start
    transcribe(asr, speed_clips["waveform"].head(1))   # warm-up

    start = time.perf_counter()
    transcribe(asr, speed_clips["waveform"])
    ms_per_clip = (time.perf_counter() - start) / len(speed_clips) * 1000

    results[name].update({"cpu_load_s": round(load_s, 2), "cpu_ms_per_clip": round(ms_per_clip, 1)})
    print(f"{name:<22} load={load_s:5.2f}s  per clip={ms_per_clip:7.1f}ms")
    del asr

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

flax_model.msgpack:   0%|          | 0.00/151M [00:00<?, ?B/s]

tf_model.h5:   0%|          | 0.00/151M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/151M [00:00<?, ?B/s]

...

whisper_base_pretrained load= 0.26s  per clip= 2880.6ms


Device set to use cpu


whisper_tiny_finetuned load= 0.46s  per clip= 1430.4ms


Device set to use cpu


whisper_base_finetuned load= 0.60s  per clip= 2843.8ms


# Step 6: Compare and pick the speech-to-text model for the app

**Selection rule** (decided before training):
1. Lowest **validation WER on lines unseen in training**: the score that rewards generalisation, not memorising the script.
2. If several models are within 0.01 of the best, choose the **fastest on CPU**, because the app runs on CPU.

Test scores are reported for every model but not used to choose.

In [10]:
comparison = pd.DataFrame(results).T.infer_objects().rename_axis("model").reset_index()

best = comparison["validation_wer_unseen"].min()
close = comparison[comparison["validation_wer_unseen"] <= best + 0.01]
SELECTED = close.sort_values("cpu_ms_per_clip").iloc[0]["model"]
comparison["selected"] = comparison["model"] == SELECTED
comparison.to_csv(f"{RESULTS_DIR}/model_comparison.csv", index=False)

print("Selected:", SELECTED)
cols = ["model", "best_epoch", "validation_wer_unseen", "test_wer_all", "test_wer_unseen", "test_wer_seen",
        "test_exact_match_all", "test_exact_match_unseen", "cpu_ms_per_clip", "selected"]
comparison[cols]

Selected: whisper_base_finetuned


,model,best_epoch,validation_wer_unseen,test_wer_all,test_wer_unseen,test_wer_seen,test_exact_match_all,test_exact_match_unseen,cpu_ms_per_clip,selected
0,whisper_tiny_pretrained,NaN,0.142300,0.154849,0.127717,0.206168,0.592523,0.564516,1394.9,False
1,whisper_base_pretrained,NaN,0.143414,0.117519,0.095109,0.159909,0.668224,0.647177,2880.6,False
2,whisper_tiny_finetuned,2.0,0.104985,0.060043,0.084843,0.013135,0.839252,0.685484,1430.4,False
3,whisper_base_finetuned,3.0,0.077137,0.042662,0.059783,0.010280,0.871028,0.745968,2843.8,True


In [11]:
# The same test lines before and after fine-tuning
cols = ["spoken_text", "seen_in_train", "whisper_base_pretrained", "whisper_base_finetuned",
        "whisper_tiny_pretrained", "whisper_tiny_finetuned"]
transcripts[cols].sample(12, random_state=SEED)

,spoken_text,seen_in_train,whisper_base_pretrained,whisper_base_finetuned,whisper_tiny_pretrained,whisper_tiny_finetuned
644,No more.,True,No more.,No more.,No more!,No more.
629,"Where should we go, rift walker? They cannot w...",False,"Where should we go, Riftwalker? They cannot wa...","Where should we go, rift walker? They cannot w...","Where should we go, Riftwalker? They cannot wa...","Where shall we go, rift walker? They cannot wa..."
70,Rotating.,True,Rotating.,Rotating.,Rotating.,Rotating.
962,Ended.,True,End it.,End it.,End it.,End it.
787,"That was worrisome, Reyna.",False,"That was worrisome, Rayna.","That was worrisome, Reyna.",That was worrisome Rayna.,"That was worrisome, Reyna."
706,"Cypher, I'm so sorry about breaking all your s...",False,"Cypher, I'm so sorry about breaking all your s...","Cypher, I'm so sorry about breaking all your s...",Sorry for I'm so sorry about breaking on your ...,"Cypher, I'm so sorry about breaking all your s..."
308,I'm faking C.,True,I'm faking sea.,I'm faking C.,I'm faking sea.,I'm faking C.
999,Well done.,True,Well done.,Well done.,Well done.,Well done.
336,"Got your back, Neon!",False,"Got your back, neon.","Got your back, Neon!","Gotcha back, neon.","Got your back, Neon."
321,We'll catch them!,False,We'll catch them.,We'll catch them.,We'll catch them!,We'll catch them.


In [12]:
# Where does the selected model still go wrong? Most frequent word errors on the test set
def top_word_errors(references, hypotheses, n=20):
    """The n most frequent substituted, deleted and inserted words."""
    substitutions, insertions, deletions = jiwer.collect_error_counts(
        jiwer.process_words(list(references), list(hypotheses))
    )
    errors = pd.DataFrame(
        [("substitution", ref, hyp, c) for (ref, hyp), c in substitutions.items()]
        + [("deletion", word, "", c) for word, c in deletions.items()]
        + [("insertion", "", word, c) for word, c in insertions.items()],
        columns=["error", "script word", "transcribed as", "count"],
    )
    errors = errors.sort_values(["count", "error", "script word", "transcribed as"], ascending=[False, True, True, True])
    return errors.head(n).reset_index(drop=True)


errors = top_word_errors(transcripts["reference"], [normalizer(t) for t in transcripts[SELECTED]])
errors.to_csv(f"{RESULTS_DIR}/{SELECTED}_top_word_errors.csv", index=False)
errors

,error,script word,transcribed as,count
0,deletion,0,,4
1,insertion,,is,4
2,substitution,em,them,4
3,insertion,,haha,2
4,insertion,,they are,2
5,substitution,are concussed,can cost,2
6,substitution,creds,cries,2
7,substitution,him,them,2
8,substitution,it,its,2
9,substitution,kay,kale,2


In [13]:
# Record exactly how these results were produced
with open(f"{RESULTS_DIR}/run_config.json", "w") as f:
    json.dump({
        "seed": SEED,
        "dataset": DATASET,
        "dataset_revision": REVISION,
        "splits_csv": {"url": SPLITS_URL, "sha256": SPLITS_SHA256},
        "clips": {split: len(df) for split, df in data.items()},
        "seen_in_train": {split: int(data[split]["seen_in_train"].sum()) for split in ["validation", "test"]},
        "candidates": CANDIDATES,
        "hyperparameters": {
            "num_epochs": NUM_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "warmup_ratio": 0.1,
            "batch_size": BATCH_SIZE,
            "fp16": device == "cuda",
            "trained": "whole model",
        },
        "decoding": "greedy, one clip at a time",
        "text_normalizer": "transformers EnglishTextNormalizer (Whisper)",
        "selection_rule": "lowest validation WER on lines unseen in training; ties within 0.01 go to lowest cpu_ms_per_clip",
        "selected_model": SELECTED,
        "versions": {"transformers": transformers.__version__, "torch": torch.__version__, "jiwer": JIWER_VERSION},
        "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
        "run_at_utc": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    }, f, indent=2)
print(open(f"{RESULTS_DIR}/run_config.json").read())

{
  "seed": 42,
  "dataset": "NgThVinh/ValorantAgentVoiceLines",
  "dataset_revision": "c2e0d3493f44da5be07090d6ebb9b69ebbcb2060",
  "splits_csv": {
    "url": "https://raw.githubusercontent.com/EthanIsaac/valorant-voiceline-qa/2837dcb342749a346d4370e63545c575ba5c2a07/results/agent_classifier/splits.csv",
    "sha256": "72b658b97258de7bc3e77a2b88908080dd46be76d0162c8905cceff2c903dd7d"
  },
  "clips": {
    "train": 8611,
    "validation": 1059,
    "test": 1070
  },
  "seen_in_train": {
    "validation": 537,
    "test": 574
  },
  "candidates": {
    "whisper_tiny": "openai/whisper-tiny.en",
    "whisper_base": "openai/whisper-base.en"
  },
  "hyperparameters": {
    "num_epochs": 3,
    "learning_rate": 1e-05,
    "warmup_ratio": 0.1,
    "batch_size": 16,
    "fp16": true,
    "trained": "whole model"
  },
  "decoding": "greedy, one clip at a time",
  "text_normalizer": "transformers EnglishTextNormalizer (Whisper)",
  "selection_rule": "lowest validation WER on lines unseen in trai

# Step 7: Upload the selected model to Hugging Face

If the selected model is a fine-tuned one, it is uploaded so the app can load it from the Hub, together with a model card.
If a pre-trained model wins, nothing is uploaded: the app uses the original model directly.

**Token:** a Hugging Face *write* token as the Colab secret `HF_TOKEN` (read in Step 0). Without a token the upload is skipped.

In [14]:
from huggingface_hub import ModelCard, repo_exists

key, version = SELECTED.rsplit("_", 1)
HF_REPO = f"eibautistat/valorant-{key.replace('_', '-')}-en"
row = comparison.set_index("model").loc[SELECTED]

card = f"""---
library_name: transformers
pipeline_tag: automatic-speech-recognition
base_model: {CANDIDATES[key]}
datasets:
- {DATASET}
language:
- en
tags:
- valorant
- whisper
metrics:
- wer
---

# Whisper for VALORANT voice lines

[`{CANDIDATES[key]}`](https://huggingface.co/{CANDIDATES[key]}) fine-tuned on the English voice lines of the 26 VALORANT agents,
so that game vocabulary (site callouts, agent names, abilities) is transcribed correctly.
Part of a voice-line QA & localization assistant for Riot Games (HKUST ISOM5240 project):
https://github.com/EthanIsaac/valorant-voiceline-qa

- **Data:** [`{DATASET}`](https://huggingface.co/datasets/{DATASET}) (revision `{REVISION[:7]}`), {len(data["train"])} training lines; split per agent, grouped by line
- **Training:** {NUM_EPOCHS} epochs, learning rate {LEARNING_RATE}, best epoch by validation WER

## Results (test set, {len(data["test"])} lines; text normalised with Whisper's English normaliser)

| Metric | Pre-trained | Fine-tuned |
|---|---|---|
| WER, all lines | {comparison.set_index("model").loc[f"{key}_pretrained", "test_wer_all"]:.3f} | {row["test_wer_all"]:.3f} |
| WER, lines not in the training set | {comparison.set_index("model").loc[f"{key}_pretrained", "test_wer_unseen"]:.3f} | {row["test_wer_unseen"]:.3f} |
| Exact match, all lines | {comparison.set_index("model").loc[f"{key}_pretrained", "test_exact_match_all"]:.1%} | {row["test_exact_match_all"]:.1%} |

## Usage

```python
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model="{HF_REPO}")
asr("voice_line.wav")["text"]
```

## Limitations

Trained on studio-quality voice lines of the existing agents; new agent names or recordings with music and effects may be transcribed less accurately.
"""

UPLOADED = False
if version != "finetuned":
    print("A pre-trained model was selected: nothing to upload.")
elif not UPLOAD_HF_MODELS:
    print("Upload skipped (UPLOAD_HF_MODELS is not true). The published model is at:", f"https://huggingface.co/{HF_REPO}")
else:
    model = WhisperForConditionalGeneration.from_pretrained(f"models/{key}")
    processor = WhisperProcessor.from_pretrained(f"models/{key}")
    commit = f"Fine-tuned {CANDIDATES[key]} on {DATASET}@{REVISION[:7]} (seed {SEED})"
    model.push_to_hub(HF_REPO, commit_message=commit)
    processor.push_to_hub(HF_REPO, commit_message=commit)
    ModelCard(card).push_to_hub(HF_REPO, commit_message="Model card")
    UPLOADED = True
    print("Uploaded:", f"https://huggingface.co/{HF_REPO}")

Upload skipped (UPLOAD_HF_MODELS is not true). The published model is at: https://huggingface.co/eibautistat/valorant-whisper-base-en


In [15]:
# Test the uploaded model exactly as the app will use it: downloaded from the Hub
if version == "finetuned" and repo_exists(HF_REPO):
    hub_asr = pipeline("automatic-speech-recognition", model=HF_REPO, device=-1)
    for _, row in data["test"].sample(3, random_state=SEED).iterrows():
        print(f"script: {row['spoken_text']!r:45} transcript: {hub_asr({'raw': row['waveform'], 'sampling_rate': TARGET_SR})['text'].strip()!r}")
else:
    print("No fine-tuned model on the Hub to test.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/356 [00:00<?, ?B/s]

Device set to use cpu


script: 'No more.'                                    transcript: 'No more.'
script: 'Where should we go, rift walker? They cannot watch everywhere at once.' transcript: 'Where should we go, rift walker? They cannot watch everywhere at once.'
script: 'Rotating.'                                   transcript: 'Rotating.'


# Save and download the results

Every file this notebook generated is listed with its **SHA-256 fingerprint** in `SHA256SUMS`.
Training files may differ slightly between runs (GPU computation is not bit-for-bit deterministic, and they include runtimes).
Then everything is zipped and downloaded as `results_speech_to_text_finetuning.zip`.

In [16]:
from pathlib import Path

result_files = sorted(p for p in Path(RESULTS_DIR).rglob("*") if p.is_file() and p.name != "SHA256SUMS")
with open(f"{RESULTS_DIR}/SHA256SUMS", "w") as f:
    for p in result_files:
        digest = hashlib.sha256(p.read_bytes()).hexdigest()
        f.write(f"{digest}  {p.relative_to(RESULTS_DIR)}\n")
        print(f"{digest[:16]}…  {p.relative_to(RESULTS_DIR)}  ({p.stat().st_size:,} bytes)")

0605a4a340866476…  model_comparison.csv  (1,618 bytes)
f6e4f6b03712539e…  run_config.json  (1,267 bytes)
58f3cc8b37fba34f…  test_transcripts.csv  (185,078 bytes)
58c31342a1ce7e55…  training_logs.json  (13,027 bytes)
ba1e186e2cc9debc…  whisper_base_finetuned_top_word_errors.csv  (478 bytes)


In [17]:
!rm -f results_speech_to_text_finetuning.zip && zip -qrX results_speech_to_text_finetuning.zip results/speech_to_text_finetuning
from google.colab import files
files.download("results_speech_to_text_finetuning.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>